<a href="https://colab.research.google.com/github/lara-ms/Aula-M06/blob/main/ATV_15_Lara_Siecola.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Nome: Lara Moreira Siécola
## Matricula: 2365
-----


# **ATV 15 — Testes Categorizados**

Esta atividade pratica os **testes para dados categorizados**: Qui‑quadrado clássico, Qui‑quadrado com correção de Yates, Teste Exato de Fisher e Teste de McNemar.

**Objetivos**
- Montar tabelas de contingência e formular hipóteses (H0/H1).
- Calcular valores **esperados** e a estatística **χ²** (ou **p‑valor**), decidir e concluir.
- Usar `scipy.stats`/`statsmodels` para conferir os cálculos.
- Interpretar resultados nos níveis usuais de significância (1%, 5%, 10%).




## Instruções gerais

- Execute as células na ordem em que aparecem.
- Sempre **declare as hipóteses** antes de calcular (H0 = igualdade; H1 = diferença).
- Para tabelas 2×2, você pode usar:
  - `stats.chi2_contingency(obs, correction=False)` → χ² clássico;
  - `stats.chi2_contingency(obs, correction=True)` → com **Yates**;
  - `stats.fisher_exact(obs, alternative="two-sided")` → **Fisher** (n pequeno ou valores esperados < 5);
  - `mcnemar(tabela_pareada, exact=True/False, correction=True/False)` → **McNemar** (amostras pareadas).

In [53]:
# Imports úteis para toda a atividade
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

pd.set_option("display.precision", 4)



## Funções auxiliares


In [54]:
def esperados_from_obs(obs):
    obs = np.asarray(obs, dtype=float)
    row_sum = obs.sum(axis=1, keepdims=True)
    col_sum = obs.sum(axis=0, keepdims=True)
    total = obs.sum()
    return row_sum @ col_sum / total

def decide_from_p(p, alphas=(0.10, 0.05, 0.01)):
    return {a: ("Rejeita H0" if p < a else "Não rejeita H0") for a in alphas}

def decide_from_chi2(chi2, crits):
    return {a: ("Rejeitar H0" if chi2 > crits[a] else "Falha em rejeitar H0") for a in crits}



## Exercício 1 — Qui‑quadrado **clássico** (amostras independentes, **n > 40**)

**Contexto:** Uma empresa de eletrônica deseja avaliar se a ocorrência de **defeitos de soldagem** em placas de circuito impresso (PCIs) está associada ao processo de montagem utilizado. Para isso, foram inspecionadas placas de dois grupos independentes: **Montagem Automatizada** e **Montagem Convencional**.

A tabela apresenta os **valores observados** (2×2), com **n = 160** placas:

| Processo de montagem | **Com defeito** | **Sem defeito** | **Total** |
|----------------------|:---------------:|:--------------:|:---------:|
| **Automatizada**     |        8        |       72       |    80     |
| **Convencional**     |       26        |       54       |    80     |
| **Total**            |       34        |      126       |   160     |

1. Formule H0 e H1 para investigar a associação entre o processo de montagem e a ocorrência de defeitos.
2. Calcule a **tabela de esperados** e a estatística **χ² calculado** (sem correção).
3. Obtenha **gl**, **χ² tabelado** para α∈{10%,5%,1%} (use SciPy) e o **p‑valor**.
4. Decida e conclua se há evidências de associação entre o processo de montagem e a presença de defeitos de soldagem.


**H0** (Hipótese Nula) -> processo de montagem e presença de defeitos **são independentes**

**H1** (Hipótese Alternativa) -> processo de montagem e presença de defeitos **NÃO são independentes**

In [55]:
# tabela dos valores observados 2x2
obs = np.array([[8, 72], [26, 54]])

# tabela de esperados
val_esperados = esperados_from_obs(obs)
print("Tabela de Esperados :\nCom Defeito   Sem Defeito")
print(pd.DataFrame(val_esperados))

# x^2 calculado
# x2 = stats.chi2_contingency(obs, correction=False)
c2cr, pvalue, dof, expf = stats.chi2_contingency(obs, correction=False)
print(f"\nχ² = {c2cr:.4f}")
print(f"p-value = {pvalue:.4f}")
print(f"gl = {dof}")

# calcular de novo porem com alpha
alpha = [0.10, 0.05, 0.01]
crit = {a : stats.chi2.ppf(1 - a, dof) for a in alpha}
print("\nχ² tabela para alpha :")
for a, val in crit.items():
  print(f"{int(a*100)}% -> {val:.4f}")

# decisoes
print("\nDecisões :")
print(f"χ² -> {decide_from_chi2(c2cr, crit)}")
print(f"p-valor -> {decide_from_p(pvalue, alpha)}")

Tabela de Esperados :
Com Defeito   Sem Defeito
      0     1
0  17.0  63.0
1  17.0  63.0

χ² = 12.1008
p-value = 0.0005
gl = 1

χ² tabela para alpha :
10% -> 2.7055
5% -> 3.8415
1% -> 6.6349

Decisões :
χ² -> {0.1: 'Rejeitar H0', 0.05: 'Rejeitar H0', 0.01: 'Rejeitar H0'}
p-valor -> {0.1: 'Rejeita H0', 0.05: 'Rejeita H0', 0.01: 'Rejeita H0'}


**Conclusão:**

Pode-se concluir que há uma forte evidência de associação, pois se é rejeitado o H0 em 10%, 5% e 1%.

## Exercício 2 — Qui‑quadrado **com correção de Yates** (amostras independentes, **20 < n < 40**)

**Contexto:** Durante um experimento com redes de sensores, uma equipe de engenharia comparou dois **protocolos de comunicação**, A e B. Em cada grupo, 17 dispositivos independentes realizaram um teste de envio de dados, sendo classificados conforme o resultado da transmissão.

A tabela apresenta os **valores observados** (2×2), com **n = 34** dispositivos:

| Protocolo | **Transmissão bem-sucedida** | **Falha na transmissão** | **Total** |
|-----------|:---------------------------:|:------------------------:|:---------:|
| **A**     |             13              |            4             |    17     |
| **B**     |              7              |           10             |    17     |
| **Total** |             20              |           14             |    34     |

1. Formule H0 e H1 para verificar se o resultado da transmissão está associado ao protocolo utilizado.
2. Calcule os **esperados** e a estatística **χ² com correção de Yates**.
3. Informe **gl**, **p‑valor** e **decisão** para α∈{10%,5%}, interpretando os resultados nos dois níveis de significância.


H0 -> resultado da transmissão independente do protocolo

H1 -> resultado da transmissão associado ao protocolo

In [56]:
# tabela dos valores observados 2x2
obs = np.array([[13, 4], [7, 10]])

# tabela dos valores esperados
val_esperados = esperados_from_obs(obs)
print("Tabela de Esperados :\nBem-sucedida   Falha")
print(pd.DataFrame(val_esperados))

# χ² com correção de Yates
c2cr, pvalue, dof, expf = stats.chi2_contingency(obs, correction=True)
print(f"\nχ² = {c2cr:.4f}")
print(f"p-value = {pvalue:.4f}")
print(f"gl = {dof}")

# valores para alpha
alpha = [0.10, 0.05]
print("\nDecisão :")
print(f"p-valor -> {decide_from_p(pvalue, alpha)}")



Tabela de Esperados :
Bem-sucedida   Falha
      0    1
0  10.0  7.0
1  10.0  7.0

χ² = 3.0357
p-value = 0.0815
gl = 1

Decisão :
p-valor -> {0.1: 'Rejeita H0', 0.05: 'Não rejeita H0'}


**Conclusão:**

Pode-se concluir que para alpha = 10% rejeita o H0, ou seja, há evidência de associação. Já para alpha = 5% não se rejeita, ou seja, não há evidência suficiente.



## Exercício 3 — **Teste Exato de Fisher** (amostras independentes, **n < 20** ou esperados < 5)

**Contexto:** Uma equipe desenvolveu sensores para monitoramento ambiental e comparou duas técnicas de **vedação de invólucros eletrônicos**: resina epóxi e silicone. Após um ensaio padronizado de exposição à água, cada protótipo foi classificado como **sem infiltração** ou **com infiltração**.

A tabela apresenta os resultados de **n = 16** protótipos independentes:

| Técnica de vedação | **Sem infiltração** | **Com infiltração** | **Total** |
|-------------------|:------------------:|:------------------:|:---------:|
| **Resina epóxi**  |         7          |         1          |     8     |
| **Silicone**      |         2          |         6          |     8     |
| **Total**         |         9          |         7          |    16     |

1. Formule H0 e H1 para investigar a associação entre a técnica de vedação e a ocorrência de infiltração (**duas caudas**).
2. Aplique o **Teste Exato de Fisher** (`alternative="two-sided"`) e obtenha o **p‑valor**.
3. Decida em α=5% e conclua sobre os resultados do ensaio.


H0 -> tecnica de vedação e infiltração são independentes

H1 -> técnica de vedação e infiltração são associados

In [57]:
# tabela de valores observados 2x2
obs = np.array([[7, 1], [2, 6]])

# so para testar tabela de esperados
val_esperados = esperados_from_obs(obs)
print("Tabela de Esperados :\nSem infiltração   Com infiltração")
print(pd.DataFrame(val_esperados))

# teste exato de fisher
pvalue = stats.fisher_exact(obs, alternative="two-sided")[1]
print(f"\np-value = {pvalue:.4f}")

# decidindo com alpha = 5%
print("\nDecisão :")
print(f"p-valor -> {decide_from_p(pvalue, [0.05])}")

Tabela de Esperados :
Sem infiltração   Com infiltração
     0    1
0  4.5  3.5
1  4.5  3.5

p-value = 0.0406

Decisão :
p-valor -> {0.05: 'Rejeita H0'}


**Conclusão:**

Pode-se concluir que os esperados são < 5 é usado o Fisher exato, p = 0.0406 < 0.05, então se é rejeitado o H0, logo há associação entre a tecnica e infiltração.

## Exercício 4 — **Teste de McNemar** (amostras pareadas: antes × depois)

**Contexto:** Uma equipe de sistemas embarcados avaliou o efeito de uma **atualização de firmware** sobre a comunicação de **44 dispositivos IoT**. Os mesmos dispositivos foram submetidos a um teste de comunicação **antes** e **depois** da atualização, sendo classificados como **Estável** ou **Instável** em cada momento.

A tabela apresenta os **dados pareados** (linhas = Antes; colunas = Depois):

| **Antes / Depois** | **Estável** | **Instável** | **Total** |
|-------------------|:----------:|:-----------:|:---------:|
| **Estável**       |     18     |      3      |    21     |
| **Instável**      |     13     |     10      |    23     |
| **Total**         |     31     |     13      |    44     |

1. Formule H0 e H1 (**duas caudas**) para avaliar se a proporção de dispositivos com comunicação estável mudou após a atualização.
2. Calcule o **Teste de McNemar** com e sem correção de continuidade de Yates; reporte a **estatística** e o **p‑valor** de cada versão.
3. Decida para α=1% e α=5% e conclua sobre o efeito da atualização na estabilidade da comunicação.


H0 -> a proporção de dispositivos estáveis não mudou

H1 -> a proporção de dispositivos estáveis mudou

In [58]:
# tabela de dados pareados
pareados = np.array([[18, 3], [13, 10]])

# teste de McNemar
sem_Yates = mcnemar(pareados, exact=False, correction=False)
com_Yates = mcnemar(pareados, exact=False, correction=True)

print("Teste de McNemar : ")
print(f"Sem Yates -> χ² = {sem_Yates.statistic:.4f} -> p-value = {sem_Yates.pvalue:.4f}")
print(f"Com Yates -> χ² = {com_Yates.statistic:.4f} -> p-value = {com_Yates.pvalue:.4f}")

# decisoes
print("\nDecisões :")
print(f"Sem Yates : p-valor -> {decide_from_p(sem_Yates.pvalue, [0.01, 0.05])}")
print(f"Com Yates : p-valor -> {decide_from_p(com_Yates.pvalue, [0.01, 0.05])}")

Teste de McNemar : 
Sem Yates -> χ² = 6.2500 -> p-value = 0.0124
Com Yates -> χ² = 5.0625 -> p-value = 0.0244

Decisões :
Sem Yates : p-valor -> {0.01: 'Não rejeita H0', 0.05: 'Rejeita H0'}
Com Yates : p-valor -> {0.01: 'Não rejeita H0', 0.05: 'Rejeita H0'}


**Conclusão:**

Pode-se concluir que sem Yates: χ² = 6,25, p = 0,0124 e com Yates: χ² = 5,0625, p = 0,0244. Em alpha = 5% rejeita-se H0 nas duas versões, ou seja, há evidências de que a atualização aumentou a estabilidade. Já em alpha = 1% não se rejeita H0 em nenhum, então a evidência não é forte o suficiente.